In [1]:
import numpy as np
import pandas as pd
from docx2md import do_convert
import re
import torch
from sentence_transformers import SentenceTransformer, util

In [2]:
np.random.seed(42)
device = torch.device("cpu")

# Globals

In [3]:
PATTERNS = {
    "main_diagnosis": r"(<table id=['\"]table1['\"]>.*?</table>)",
    "other_diagnoses": r"Weitere bekannte Erkrankungen:(.*?)(?=Was wurde bisher gemacht\?)",
    "summary": r"Was wurde bisher gemacht\?(.*?)(?=Was sollten Sie jetzt tun\?)",
    "procedures": r"(<table id=['\"]table2['\"]>.*?</table>)"
}

# Functions

In [4]:
def extract_docx(ID, AI):
    """
    Extract the text content from a DOCX file based on the given document ID.

    Parameters
    ----------
    ID : str or int
        Unique identifier corresponding to a DOCX file.
    AI : bool
        If True, the DOCX file is read from `./GISelA_translations/`.
        If False, the DOCX file is read from `./Doctor_translations/`.

    Returns
    -------
    str
        The extracted plain text from the DOCX file.

    Notes
    -----
    This function expects a helper function `do_convert(filename)` to be defined
    elsewhere, which handles the actual DOCX-to-text conversion.
    """
    base_path = "./GISelA_translations" if AI else "./Doctor_translations"
    filename = f"{base_path}/{ID}.docx"
    return do_convert(filename)


def slice_text(text, patterns=PATTERNS):
    """
    Extract text segments from a document using regex patterns.

    Parameters
    ----------
    text : str
        The full document text to be segmented.
    patterns : dict
        A dictionary mapping segment names to their regex extraction patterns.

    Returns
    -------
    dict
        A dictionary where each key corresponds to a section name and each value
        contains the extracted text (or None if not found).
    """
    segments = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text, flags=re.DOTALL)
        if match:
            value = match.group(1).strip()
            segments[key] = value
        else:
            segments[key] = None
    return segments


def slice_table(table):
    """
    Parse an HTML-like table string into a list of (key, value) pairs.

    Parameters
    ----------
    table : str
        String containing HTML table markup (<table>...</table>).

    Returns
    -------
    list of tuple of (str, str)
        Each tuple represents a row, with the first and second <td> cell values
        stripped of tags and whitespace.
    """
    rows = re.findall(r"<tr.*?>(.*?)</tr>", table, flags=re.DOTALL)
    pairs = []

    for row in rows:
        tds = re.findall(r"<td.*?>(.*?)</td>", row, flags=re.DOTALL)
        a = re.sub(r"<.*?>", "", tds[0]).strip()
        b = re.sub(r"<.*?>", "", tds[1]).strip()
        pairs.append((a, b))

    return pairs


def slice_other_diagnoses(segment):
    """
    Extract pairs of known conditions and their details from a text segment.

    Parameters
    ----------
    segment : str
        Text section containing diagnoses formatted like 'Condition (Detail)'.

    Returns
    -------
    list of tuple of (str, str)
        A list of (condition, detail) tuples with whitespace stripped.
    """
    matches = re.findall(r"([^\n(]+)\(([^)]+)\)", segment)
    return [(a.strip(), b.strip()) for a, b in matches]


def tuples_to_string(data):
    """
    Convert a list of (title, text) tuples into a multi-line string
    containing only the text values.

    The first tuple in the list is skipped. For each remaining tuple,
    only the second element (text) is extracted. These text values are
    joined together into a single string, separated by newline characters.

    Parameters
    ----------
    data : list of tuple of (str, str)
        A list of (title, text) tuples where the first tuple is ignored.

    Returns
    -------
    str
        A single string containing the text from each tuple (excluding
        the first), separated by newline characters.
    """
    return "\n".join(f"{text}" for title, text in data[1:])


def preprocess_data(ID, AI):
    """
    Extract, preprocess, and format data from a DOCX document into text segments.

    Steps
    -----
    1. Extract text from the DOCX file identified by `ID` from the appropriate
       source based on `AI`.
    2. Slice the text into predefined sections using regex patterns.
    3. Apply postprocessing functions to each relevant section (e.g. parse tables).
    4. Convert tuple-based sections into formatted multi-line strings.

    Parameters
    ----------
    ID : str or int
        Unique identifier of the document to be processed.
    AI : bool
        If True, extract text from the AI-generated translations directory.
        If False, extract text from the doctor-provided translations directory.

    Returns
    -------
    dict
        A dictionary of processed text segments. Each key corresponds to a
        document section (e.g., 'main_diagnosis', 'procedures'), and each
        value is a formatted string or None if the section is missing.
    """
    text = extract_docx(ID, AI)
    segments = slice_text(text)

    postprocessors = {
        "main_diagnosis": slice_table,
        "other_diagnoses": slice_other_diagnoses,
        "procedures": lambda x: slice_table(x)[1:]
    }

    for key, func in postprocessors.items():
        segments[key] = func(segments[key]) if segments.get(key) else None
        segments[key] = tuples_to_string(segments[key]) if segments[key] else None

    return segments


def combine_sections(data):
    """
    Combine a dictionary with four sections into a formatted multi-line string.

    Parameters
    ----------
    data : dict
        Dictionary with exactly four key–value pairs in the following order:
        Hauptdiagnosen, Nebendiagnosen, Epikrise, Procedere.

    Returns
    -------
    str
        A formatted string containing all sections separated by blank lines.
    """
    values = list(data.values())

    return (
        "Hauptdiagnosen\n"
        f"{values[0]}\n\n"
        "Nebendiagnosen\n"
        f"{values[1]}\n\n"
        "Epikrise\n"
        f"{values[2]}\n\n"
        "Procedere\n"
        f"{values[3]}"
    )


def clean(x):
    """
    Strip leading and trailing spaces and newline characters from a string.

    Parameters
    ----------
    x : str or any
        Input value to be cleaned.

    Returns
    -------
    str or any
        The cleaned string if `x` is a string; otherwise the input is
        returned unchanged (e.g. for None or NaN values).
    """
    return x.strip(" \n") if isinstance(x, str) else x


def get_original_report(df, ID):
    """
    Build the original formatted report string from a DataFrame row.

    The row is identified via the 'ID' column. Selected text columns are
    cleaned and combined into a coherent multi-section report.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the report data, including an 'ID' column.
    ID : str or int
        Identifier used to select the row.

    Returns
    -------
    str
        A formatted report string containing Hauptdiagnosen, Nebendiagnosen,
        Epikrise, and Procedere.
    """
    row = df.loc[df["ID"] == ID].iloc[0]

    sections = {
        "Hauptdiagnosen": clean(row["Hauptdiagnosen"]),
        "Nebendiagnosen": clean(row["Nebendiagnosen"]),
        "Epikrise": clean(row["Epikrise"]),
        "Procedere": clean(row["Procedere"]),
    }

    return combine_sections(sections)
    

def get_translated_report(ID, AI):
    """
    Generate a fully formatted report string from a DOCX translation.

    This function extracts, preprocesses, and formats the text sections
    of a DOCX document into a coherent report. The source of the document
    is determined by the `AI` flag.

    Parameters
    ----------
    ID : str or int
        Unique identifier of the document to be processed.
    AI : bool
        If True, use the AI-generated translation (./GISelA_translations/).
        If False, use the doctor-provided translation (./Doctor_translations/).

    Returns
    -------
    str
        A formatted report containing Hauptdiagnosen, Nebendiagnosen,
        Epikrise, and Procedere.
    """
    segments = preprocess_data(ID, AI)
    report = combine_sections(segments)

    return report


def report_cosine_similarity(model, report1, report2):
    """
    Compute the cosine similarity score between two report texts using a SentenceTransformer model.

    Parameters
    ----------
    model : sentence_transformers.SentenceTransformer
        Pretrained SentenceTransformer model used to encode the texts.
    report1 : str
        First report text.
    report2 : str
        Second report text.

    Returns
    -------
    float
        Cosine similarity score between `report1` and `report2`. Value ranges from -1 (opposite) to 1 (identical).
    """
    emb1 = model.encode(report1, convert_to_tensor=True)
    emb2 = model.encode(report2, convert_to_tensor=True)

    # Compute cosine similarity and return scalar
    score = util.cos_sim(emb1, emb2).item()
    return score

# Load Data 

In [5]:
letters = pd.read_excel('./letters.xlsx', engine='openpyxl')
analysis = pd.read_excel('./study_data_sheet.xlsx', engine='openpyxl')

# Analysis

## Sentence Transformer

In [6]:
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B",
    trust_remote_code=True,
    device='cpu',
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Loop

In [7]:
%%capture

for ID in letters['ID']:
    original = get_original_report(letters, ID)
    AI_translation = get_translated_report(ID, AI=True)
    human_translation = get_translated_report(ID, AI=False)

    # Text length
    analysis.loc[analysis['ID']==ID, 'original_length'] = len(original)
    analysis.loc[analysis['ID']==ID, 'AI_translation_length'] = len(AI_translation)
    analysis.loc[analysis['ID']==ID, 'human_translation_length'] = len(human_translation)

    # Cosine Similarity: Original versus AI translation
    cosine_OG_vs_AI = report_cosine_similarity(model, original, AI_translation)
    analysis.loc[analysis['ID']==ID, 'cosine_similarity_original_vs_AI'] = cosine_OG_vs_AI
                    
    # Cosine Similarity: Original versus human translation
    cosine_OG_vs_human = report_cosine_similarity(model, original, human_translation)
    analysis.loc[analysis['ID']==ID, 'cosine_similarity_original_vs_human'] = cosine_OG_vs_human
                    
    # Cosine Similarity: AI translation versus human translation
    cosine_AI_vs_human = report_cosine_similarity(model, AI_translation, human_translation)
    analysis.loc[analysis['ID']==ID, 'cosine_similarity_AI_vs_human'] = cosine_AI_vs_human

## Save Data

In [8]:
analysis.to_csv('./analysis.csv', index=False)